# Shared symbolic basis and coefficient training

This notebook fits a single pressure model and a single composition model using all six development systems. The shared symbolic forms are defined in `script/shared_model.py`. Discovery notebooks document the preceding system-specific searches, but do not automatically construct a different shared library.

No data or predictions are bundled. First prepare your own inputs (see `data/README.md`) or run the synthetic demonstration in README.md. Set `RUN_TRAINING = True` to train new models. The monitored holdout is used for parameter selection and is not an independent test set.

**Data required.** This code-only release contains no measurements, predictions or example figures. Prepare inputs using `data/README.md` before running the cells. For a software-only check, generate explicitly fictional inputs with `python -m script.synthetic_demo` as described in the main README.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "configs/search_settings.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/search_settings.json").exists():
    raise RuntimeError("Open this notebook from the repository or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from script import workflow
from script.data import load_preos_predictions
from script.plotting import plot_system
from IPython.display import display, Image
print("Repository:", ROOT)


In [ ]:
RUN_TRAINING = False
EPOCHS = 2500
LEARNING_RATE = 5e-3
SEED = 42
PREDICTIONS = ROOT / "outputs/proposed_predictions.csv"


## Joint training
Adam optimizes the shared internal constants and carbon-number coefficient network. New runs save model weights, normalization constants, settings, coefficient values, fifth-degree coefficient polynomials and predictions.

In [ ]:
if RUN_TRAINING:
    run_dir = workflow.train_shared(epochs=EPOCHS, lr=LEARNING_RATE, seed=SEED)
    PREDICTIONS = run_dir / "predictions.csv"
    print(run_dir)


## Plot all six systems
Each PDF contains separate temperature panels. No combined-isotherm or critical-locus plot is generated.

In [ ]:
from script.data import SYSTEMS
for system in SYSTEMS:
    path = plot_system(system, predictions=PREDICTIONS, output_dir=ROOT / "outputs/figures")
    print(path)


In [ ]:
display(Image(filename=str(ROOT / "outputs/figures/C5N2_vle_comparison.png")))
